# jev-gc walkthrough: Strands + Gemini dummy agent

This notebook runs the same agent as `dummy_agent.py` **once**, capturing every OpenTelemetry span it emits, then compares two ways of building the next prompt's context from that same span history:

1. **Naive / no GC**: concatenate every tool/model span's output verbatim, growing unbounded turn over turn.
2. **jev-gc**: run the same spans through the prefilter → scorer → policy → store pipeline and call `build_context(task, budget_tokens)`.

Both numbers come from the *same* agent run — we only run Gemini/Jev once and derive both views from the captured `SpanRecord`s afterward, to keep this notebook cheap to re-run.

Requires `GEMINI_API_KEY` (real LLM, required) and optionally `JEV_API_KEY` (falls back to `FakeJevClient` if unset).

In [1]:
import os
import sys

sys.path.insert(0, ".")  # run from examples/strands_dummy_agent/

from dummy_agent import PROMPTS, build_agent, build_jevgc
from strands.telemetry import StrandsTelemetry

from jevgc.context_builder import estimate_tokens
from jevgc.otel.processor import JevGCSpanProcessor

assert "GEMINI_API_KEY" in os.environ, "export GEMINI_API_KEY before running this notebook"

## 1. Wire up jev-gc and a raw span capture, then run the agent once

In [2]:
captured_spans = []  # every SpanRecord jev-gc ever saw, in order

gc = build_jevgc()

async def _capture_and_observe(record):
    captured_spans.append(record)
    await gc.observe(record)

# StrandsTelemetry(), called with no tracer_provider, both creates a
# TracerProvider *and* registers it as the OTel global -- Strands' Agent
# binds its tracer to whatever the global provider is at construction time.
telemetry = StrandsTelemetry()
processor = JevGCSpanProcessor(on_span_record=_capture_and_observe, turn_index_fn=lambda _span: gc._turn_index)
telemetry.tracer_provider.add_span_processor(processor)

agent = build_agent()

async def run_agent():
    for turn, prompt in enumerate(PROMPTS):
        gc.advance_turn(turn)
        try:
            response = await agent.invoke_async(prompt)
            print(f"turn {turn}: {prompt}\n  -> {response}\n")
        except Exception as exc:
            print(f"turn {turn} raised (continuing): {exc}\n")
    await processor.wait_all()

await run_agent()
print(f"captured {len(captured_spans)} spans")

turn 0 raised (continuing): You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash
Please retry in 1.746735054s.

turn 1 raised (continuing): You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash
Please retry in 54.903540885s.

turn 2 raised (continuing): You exceeded your current quota, please check your plan and billing details. For more information on 

## 2. Naive baseline: what the next prompt looks like with no GC at all

Every span's raw output, concatenated, growing without bound as the conversation continues.

In [ ]:
def naive_context(spans):
    parts = []
    for span in spans:
        body = span.output_preview or span.error_message or span.input_preview or ""
        parts.append(f"[{span.name}] {body}")
    return "\n\n".join(parts)

print("turn-by-turn naive context size (tokens):")
running = []
for span in captured_spans:
    running.append(span)
    print(f"  after {span.name}: {estimate_tokens(naive_context(running))} tokens ({len(running)} spans)")

naive_text = naive_context(captured_spans)
naive_tokens = estimate_tokens(naive_text)
print(f"\nfinal naive context: {naive_tokens} tokens")

## 3. jev-gc's view of the same spans

Every span already flowed through `gc.observe` during the run above. `build_context` assembles the budget-fit result from whatever's currently HOT/WARM in the store.

In [ ]:
BUDGET = 800
gc_context = gc.build_context(task="Summarize what we learned about Tokyo's weather", budget_tokens=BUDGET)
gc_tokens = estimate_tokens(gc_context)

print(f"jev-gc context ({gc_tokens} tokens, budget={BUDGET}):\n")
print(gc_context)

## 4. A few decisions, with their `reason` field

This is the audit trail: why each span ended up where it did.

In [ ]:
from jevgc.models import Tier

for tier in (Tier.HOT, Tier.WARM, Tier.COLD):
    items = gc._store.list_by_tier(tier)
    print(f"--- {tier.value.upper()} ({len(items)} items) ---")
    for item in items[:5]:
        d = item.decision
        print(f"  {item.span_id}: treatment={d.treatment.value} used_jev={d.used_jev} reason={d.reason!r}")

## 5. The comparison

See the main [README](../../README.md#does-this-actually-matter) for why this gap only widens as a session gets longer -- this notebook's 4-turn demo is intentionally small (free-tier friendly), so the absolute numbers here are modest; the point is the *shape* of the curve, not this one session's totals.

In [ ]:
stats = gc.stats()
print(f"naive (no GC):  {naive_tokens} tokens across {len(captured_spans)} spans")
print(f"jev-gc:         {gc_tokens} tokens (budget={BUDGET})")
if naive_tokens:
    print(f"reduction:      {100 * (1 - gc_tokens / naive_tokens):.1f}%")
print()
print(stats.model_dump_json(indent=2))

await gc.aclose()